## Install

In [1]:
!pip install -q -U "trl>=0.15" peft transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 152.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.6 MB/s eta 0:00:00


## Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/spider_data.zip" /content/
!unzip -q /content/spider_data.zip -d /content/

Mounted at /content/drive


## Config + Imports

In [3]:
import os, re, time, json, sqlite3, torch
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SPIDER   = "/content/spider_data"
DB_DIR   = f"{SPIDER}/database"
TRAIN_JSON = f"{SPIDER}/train_spider.json"   # TRAIN split
N_TRAIN = 7000     # full training split

## Verifier + Reward

In [4]:
FORBIDDEN = ("drop","delete","update","insert","alter","create","replace","attach","pragma")

def build_schema_string(db_path, max_tables=20):
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True); cur = con.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    tables = [r[0] for r in cur.fetchall()][:max_tables]; out = []
    for t in tables:
        cur.execute(f'PRAGMA table_info("{t}")'); out.append(f"{t}({', '.join(r[1] for r in cur.fetchall())})")
    con.close(); return "\n".join(out)

def extract_sql(t):
    t = t if isinstance(t, str) else t[0]["content"]
    m = re.search(r"```sql\s*(.*?)```", t, re.DOTALL|re.IGNORECASE) or re.search(r"```\s*(.*?)```", t, re.DOTALL)
    return re.sub(r"\s+"," ",(m.group(1) if m else t).strip().rstrip(";").strip())

def safe_execute(db_path, sql, timeout=5.0):
    if not sql or any(k in sql.lower() for k in FORBIDDEN): return None
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=timeout)
    s=time.time(); con.set_progress_handler(lambda:(time.time()-s)>timeout,10000)
    try: return con.cursor().execute(sql).fetchall()
    except Exception: return None
    finally: con.close()

def exec_match(db_path, pred, gold):
    p = safe_execute(db_path, pred)
    if p is None: return 0.0
    g = safe_execute(db_path, gold)
    if g is None: return 0.0
    return 1.0 if set(map(tuple,p))==set(map(tuple,g)) else 0.0

def sql_reward(completions, gold_sql, db_path, **kwargs):
    return [exec_match(dbp, extract_sql(c), gold) for c,gold,dbp in zip(completions, gold_sql, db_path)]

## Build the TRAIN dataset + gate check

In [5]:
raw = json.load(open(TRAIN_JSON))
PROMPT = ("Database schema:\n{schema}\n\nWrite a single SQLite query that answers the question. "
          "Return ONLY the query inside a ```sql code block.\n\nQuestion: {q}")
rows = []
for ex in raw:
    dbp = f"{DB_DIR}/{ex['db_id']}/{ex['db_id']}.sqlite"
    if not os.path.exists(dbp): continue
    msg = [{"role":"user","content": PROMPT.format(schema=build_schema_string(dbp), q=ex["question"])}]
    rows.append({"prompt": msg, "gold_sql": ex["query"], "db_path": dbp})
    if len(rows) >= N_TRAIN: break
dataset = Dataset.from_list(rows)

# GATE: gold must score 1.0 before spending any GPU time
assert exec_match(rows[0]["db_path"], rows[0]["gold_sql"], rows[0]["gold_sql"]) == 1.0, "Verifier broken"
print(f"{len(dataset)} train examples. Verifier OK.")

7000 train examples. Verifier OK.


In [6]:
import trl, inspect
from trl import GRPOConfig
print("TRL version:", trl.__version__)
print([f for f in GRPOConfig.__dataclass_fields__ if "length" in f or "prompt" in f or "completion" in f])

TRL version: 1.9.0
['length_column_name', 'max_completion_length', 'vllm_max_model_length', 'mask_truncated_completions', 'log_completions', 'num_completions_to_print', 'log_unique_prompts', 'log_completions_hub_repo']


## Train (GRPO + LoRA)


In [7]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.9 MB/s eta 0:00:00


In [8]:
peft_config = LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

cfg = GRPOConfig(
    output_dir="grpo-sql", per_device_train_batch_size=8, gradient_accumulation_steps=2,
    num_generations=8, max_completion_length=256,
    learning_rate=1e-5, beta=0.04, temperature=0.9,
    logging_steps=1, num_train_epochs=1, bf16=True, report_to="none", save_strategy="no",
)
trainer = GRPOTrainer(model=MODEL_ID, args=cfg, train_dataset=dataset,
                      reward_funcs=sql_reward, peft_config=peft_config)
trainer.train()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,-0.033333
2,0.420560
3,0.350514
4,0.367758
5,-0.002454
6,0.421879
7,0.757803
8,0.499065
9,0.550475
10,0.150269


TrainOutput(global_step=3500, training_loss=0.03484269865708328, metrics={'train_runtime': 30542.366, 'train_samples_per_second': 0.229, 'train_steps_per_second': 0.115, 'total_flos': 0.0, 'train_loss': 0.03484269865708328, 'epoch': 1.0})

## Merge LoRA and save the trained model

In [9]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("/content/grpo-sql-merged")
trainer.tokenizer.save_pretrained("/content/grpo-sql-merged")
print("Merged model saved. Now point your baseline eval cells (6-8 + EX scorer) at this path.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AttributeError: 'GRPOTrainer' object has no attribute 'tokenizer'

In [10]:
trainer.processing_class.save_pretrained("/content/grpo-sql-merged")
print("Tokenizer saved.")

Tokenizer saved.


In [11]:
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained("/content/grpo-sql-merged")

('/content/grpo-sql-merged/tokenizer_config.json',
 '/content/grpo-sql-merged/chat_template.jinja',
 '/content/grpo-sql-merged/tokenizer.json')

In [12]:
import json
json.dump(trainer.state.log_history,
          open("/content/drive/MyDrive/spider_baseline/grpo_7k_trainlog.json", "w"))

## Load the trained model

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MERGED = "/content/grpo-sql-merged"

tok = AutoTokenizer.from_pretrained(MERGED)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MERGED, torch_dtype=torch.bfloat16, device_map="cuda").eval()
DEVICE = "cuda"

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Build the dev prompts

In [14]:
!nvidia-smi
!pip install -q -U transformers accelerate datasets sqlparse nltk tqdm

Thu Jul 23 06:08:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             58W /  400W |   14832MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [15]:
from tqdm.auto import tqdm

PROMPT = ("Database schema:\n{schema}\n\nWrite a single SQLite query that answers "
          "the question. Return ONLY the query inside a ```sql code block.\n\nQuestion: {question}")
DEV_JSON = f"{SPIDER}/dev.json"
dev = json.load(open(DEV_JSON))
examples = []
for ex in tqdm(dev):
    dbp = f"{DB_DIR}/{ex['db_id']}/{ex['db_id']}.sqlite"
    msg = [{"role":"user","content": PROMPT.format(
        schema=build_schema_string(dbp), question=ex["question"])}]
    text = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    examples.append({"text": text, "gold": ex["query"], "db_id": ex["db_id"], "db_path": dbp})
print(len(examples), "dev examples")

  0%|          | 0/1034 [00:00<?, ?it/s]

1034 dev examples


## Batched generation helper, then the greedy pass

In [16]:
def generate_batched(texts, max_new=256, do_sample=False, num_return=1, temp=0.8, bs=16):
    outs = []
    for i in tqdm(range(0, len(texts), bs)):
        enc = tok(texts[i:i+bs], return_tensors="pt", padding=True,
                  truncation=True, max_length=1536).to(DEVICE)
        kw = dict(max_new_tokens=max_new, pad_token_id=tok.pad_token_id,
                  num_return_sequences=num_return)
        kw.update(dict(do_sample=True, temperature=temp, top_p=0.95) if do_sample else dict(do_sample=False))
        with torch.no_grad():
            g = model.generate(**enc, **kw)
        g = g[:, enc["input_ids"].shape[1]:]
        outs.extend(tok.batch_decode(g, skip_special_tokens=True))
    return outs

greedy = generate_batched([e["text"] for e in examples], do_sample=False, bs=16)
preds  = [extract_sql(g) or "SELECT 1" for g in greedy]

  0%|          | 0/65 [00:00<?, ?it/s]

## Write the prediction and gold files (aligned, one line each)

In [17]:
with open("/content/gold.txt","w") as gf, open("/content/pred.txt","w") as pf:
    for e, p in zip(examples, preds):
        gf.write(f"{e['gold']}\t{e['db_id']}\n")
        pf.write(f"{p}\n")
print("wrote gold.txt and pred.txt")

wrote gold.txt and pred.txt


## pass@1 vs pass@8 diagnostic

In [19]:
TABLES = f"{SPIDER}/tables.json"
print(TABLES, os.path.exists(TABLES))   # want True

/content/spider_data/tables.json True


In [21]:
!git clone -q https://github.com/taoyds/spider /content/spider
!ls /content/spider/evaluation.py    # confirm it's there

/content/spider/evaluation.py


In [23]:
import subprocess, tempfile, os

K = 8
# reuse your `examples` (full dev, 1034) built from the merged model's tokenizer

def official_ex(pred_lines):
    """Write a pred file, score with the Spider evaluator, return overall EX."""
    with open("/content/_passk_pred.txt", "w") as f:
        f.write("\n".join(pred_lines) + "\n")
    out = subprocess.run(
        ["python", "evaluation.py", "--gold", "/content/gold.txt",
         "--pred", "/content/_passk_pred.txt", "--db", DB_DIR,
         "--table", TABLES, "--etype", "exec"],
        cwd="/content/spider", capture_output=True, text=True).stdout
    for line in out.splitlines():
        if line.strip().startswith("execution"):
            return float(line.split()[-1])   # 'all' column
    return None

# pass@1: greedy predictions you already generated → this equals your official EX
p1 = official_ex(preds)

# pass@8: sample K per example, mark an example correct if ANY sample is EX-correct
# generate K sampled completions per dev example (temperature on)
sampled = generate_batched([e["text"] for e in examples],
                           do_sample=True, num_return=K, temp=0.8, bs=8)

# build K prediction files (the j-th sample of every example), score each,
# then an example counts if it's correct in ANY of the K files
correct_any = [False] * len(examples)
for j in range(K):
    col = [extract_sql(sampled[i*K + j]) or "SELECT 1" for i in range(len(examples))]
    # per-example correctness via the official evaluator requires per-line scoring;
    # simplest reliable route: reuse exec_match here ONLY for the "any" aggregation
    for i, e in enumerate(examples):
        if not correct_any[i] and exec_match(e["db_path"], col[i], e["gold"]) >= 1:
            correct_any[i] = True
pk = sum(correct_any) / len(examples)

print(f"pass@1 (official EX, full dev): {p1:.3f}")
print(f"pass@{K} (full dev): {pk:.3f}")

  0%|          | 0/130 [00:00<?, ?it/s]

TypeError: unsupported format string passed to NoneType.__format__

In [24]:
print(f"pass@{K} (full dev): {pk:.3f}")

pass@8 (full dev): 0.809


## Clone the official evaluators

In [25]:
!git clone -q https://github.com/taoyds/spider
!git clone -q https://github.com/taoyds/test-suite-sql-eval

fatal: destination path 'spider' already exists and is not an empty directory.


## EX: the reportable execution accuracy + exact match, by difficulty

In [26]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
TABLES = f"{SPIDER}/tables.json"
print(TABLES, os.path.exists(TABLES))   # should print the path and True

In [27]:
!cd spider && python evaluation.py --gold /content/gold.txt --pred /content/pred.txt --db {DB_DIR} --table {TABLES} --etype all

medium pred: SELECT T1.Song_Name, T1.Song_release_year FROM singer AS T1 WHERE T1.Age = ( SELECT MIN(Age) FROM singer )
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:1
medium pred: SELECT s.Country, COUNT(s.Singer_ID) AS Number_of_Singers FROM singer s GROUP BY s.Country
medium gold: SELECT country ,  count(*) FROM singer GROUP BY country

eval_err_num:2
medium pred: SELECT MAX(Capacity) AS Max_Capacity, AVG(Average) AS Avg_Average FROM stadium
medium gold: select max(capacity), average from stadium

medium pred: SELECT T1.Name, T1.Capacity FROM stadium AS T1 WHERE T1.Average = ( SELECT MAX(Average) FROM stadium )
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1

medium pred: SELECT T1.Name, T1.Capacity FROM stadium AS T1 WHERE T1.Average = ( SELECT MAX(Average) FROM stadium )
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1

eval_err_num:3
medium pred: SELECT s.Name, COUNT

## TS: test-suite accuracy (do this after EX works)

In [28]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [29]:
# After placing test_suite_database.zip in your Drive root:
!cp "/content/drive/MyDrive/test_suite_database.zip" /content/
!unzip -q /content/test_suite_database.zip -d /content/
!ls /content/test_suite_database | head     # folder of <db_id>/<db_id>.sqlite, like the regular DBs

ls: cannot access '/content/test_suite_database': No such file or directory


In [30]:
!unzip -q /content/test_suite_database.zip -d /content/
!ls /content/test_suite_database | head

replace /content/database/voter_1/voter_1v515patch1.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
ls: cannot access '/content/test_suite_database': No such file or directory


In [31]:
!ls /content/                              # what got unzipped, and under what name?
!ls /content/test_suite_database | head

database  grpo-sql-merged  sample_data	    test_suite_database.zip
drive	  __MACOSX	   spider	    test-suite-sql-eval
gold.txt  _passk_pred.txt  spider_data
grpo-sql  pred.txt	   spider_data.zip
ls: cannot access '/content/test_suite_database': No such file or directory


In [32]:
!rm -rf /content/ts_db
!mkdir -p /content/ts_db
!unzip -q -o /content/test_suite_database.zip -d /content/ts_db
!ls /content/ts_db

database  __MACOSX


In [33]:
!ls /content/ts_db
!ls /content/ts_db/database | head    # if there's a 'database' wrapper

database  __MACOSX
academic
advising
atis
battle_death
car_1
concert_singer
course_teach
cre_Doc_Template_Mgt
dog_kennels
employee_hire_evaluation


In [34]:
TS_DB = "/content/ts_db/database"      # ← correct this to whatever step 2 showed
!cd test-suite-sql-eval && python evaluation.py --gold /content/gold.txt --pred /content/pred.txt --db {TS_DB} --table {TABLES} --etype exec

/content/test-suite-sql-eval/exec_eval.py:127: SyntaxWarning: invalid escape sequence '\s'
  "YEAR\s*\(\s*CURDATE\s*\(\s*\)\s*\)\s*", "2020", query, flags=re.IGNORECASE
/content/test-suite-sql-eval/parse.py:57: SyntaxWarning: invalid escape sequence '\d'
  float_nums = re.findall("[-+]?\d*\.\d+", query)
/content/test-suite-sql-eval/parse.py:62: SyntaxWarning: invalid escape sequence '\d'
  int_nums = [i.strip() for i in re.findall("[^tT]\d+", query)]
/content/test-suite-sql-eval/parse.py:70: SyntaxWarning: invalid escape sequence '\d'
  table = re.findall("[Tt]\d+\.", tok)
/content/test-suite-sql-eval/parse.py:206: SyntaxWarning: invalid escape sequence '\.'
  for table, col, val1, val2 in re.findall('(?:([^\.\s]*)\.)?([^\.\s]+) between ([^\s;]+) and ([^\s;]+)', query, re.IGNORECASE):
                     easy                 medium               hard                 extra                all                 
count                248                  446                  174            

In [35]:
!cd test-suite-sql-eval && python evaluation.py \
    --gold /content/gold.txt --pred /content/pred.txt \
    --db {DB_DIR} --table {TABLES} --etype exec

                     easy                 medium               hard                 extra                all                 
count                248                  446                  174                  166                  1034                
=====================   EXECUTION ACCURACY     =====================
execution            0.871                0.740                0.603                0.422                0.697               


## Save everything to Drive

In [36]:
!mkdir -p "/content/drive/MyDrive/spider_baseline/models"
!cp -r /content/grpo-sql-merged "/content/drive/MyDrive/spider_baseline/models/grpo_7k_merged"
!du -sh "/content/drive/MyDrive/spider_baseline/models/grpo_7k_merged"

5.8G	/content/drive/MyDrive/spider_baseline/models/grpo_7k_merged


In [40]:
import shutil, json, datetime, os
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = f"/content/drive/MyDrive/spider_grpo_fullrun/{stamp}"
os.makedirs(SAVE_DIR, exist_ok=True)

for f in ["/content/gold.txt", "/content/pred.txt"]:
    if os.path.exists(f): shutil.copy(f, SAVE_DIR)
json.dump(greedy, open(f"{SAVE_DIR}/raw_greedy.json", "w"))

summary = {
    "model": MODEL_ID, "n_train": 7000, "n_dev": len(examples),
    "EX": {"easy":0.847,"medium":0.455,"hard":0.408,"extra":0.223,"all":0.504},
    "exact_match_all": 0.423,
    "IUEN_acc": {"hard":0.143,"extra":0.059,"all":0.105},
    "ts_eval_single_db": 0.697, "ts_eval_testsuite_db": 0.607, "fp_gap": 0.090,
    "pass@1_loose_fulldev": round(p1_loose,4), "pass@8_loose_fulldev": round(pk,4),
    "timestamp": stamp,
}
json.dump(summary, open(f"{SAVE_DIR}/summary.json","w"), indent=2)
print(SAVE_DIR); print(summary)

/content/drive/MyDrive/spider_grpo_fullrun/20260723_065743
{'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct', 'n_train': 7000, 'n_dev': 1034, 'EX': {'easy': 0.847, 'medium': 0.455, 'hard': 0.408, 'extra': 0.223, 'all': 0.504}, 'exact_match_all': 0.423, 'IUEN_acc': {'hard': 0.143, 'extra': 0.059, 'all': 0.105}, 'ts_eval_single_db': 0.697, 'ts_eval_testsuite_db': 0.607, 'fp_gap': 0.09, 'pass@1_loose_fulldev': 0.677, 'pass@8_loose_fulldev': 0.8085, 'timestamp': '20260723_065743'}


In [39]:
hits = sum(exec_match(e["db_path"], p, e["gold"]) for e, p in zip(examples, preds))
p1_loose = hits/len(examples)
print(f"pass@1 (loose, full dev): {p1_loose:.3f}   pass@8: {pk:.3f}")

pass@1 (loose, full dev): 0.677   pass@8: 0.809


## Additional


Cell A — stash the 7K results

In [41]:
grpo_preds, grpo_greedy = preds, greedy
grpo_p1, grpo_p8 = p1_loose, pk
print("stashed:", grpo_p1, grpo_p8)

stashed: 0.6769825918762089 0.8085106382978723


Cell B — load the base model

In [42]:
del model
torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda").eval()
print("base model loaded")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

base model loaded


Cell C — base pass@1 (greedy, full dev, loose checker)

In [43]:
base_greedy = generate_batched([e["text"] for e in examples], do_sample=False, bs=16)
base_preds  = [extract_sql(g) or "SELECT 1" for g in base_greedy]

hits = sum(exec_match(e["db_path"], p, e["gold"]) for e, p in zip(examples, base_preds))
base_p1 = hits/len(examples)
print(f"BASE pass@1 (loose, full dev): {base_p1:.3f}")

  0%|          | 0/65 [00:00<?, ?it/s]

BASE pass@1 (loose, full dev): 0.496


Cell D — base pass@8

In [44]:
K = 8
base_samp = generate_batched([e["text"] for e in examples], do_sample=True,
                             num_return=K, temp=0.8, bs=8)
correct_any = [False]*len(examples)
for i, e in enumerate(examples):
    for j in range(K):
        if exec_match(e["db_path"], extract_sql(base_samp[i*K+j]), e["gold"]) >= 1:
            correct_any[i] = True; break
base_p8 = sum(correct_any)/len(examples)
print(f"BASE pass@8 (loose, full dev): {base_p8:.3f}")

  0%|          | 0/130 [00:00<?, ?it/s]

BASE pass@8 (loose, full dev): 0.733


Cell E — the comparison + save

In [45]:
print(f"{'':10} {'pass@1':>8} {'pass@8':>8}")
print(f"{'base':10} {base_p1:>8.3f} {base_p8:>8.3f}")
print(f"{'GRPO 7K':10} {grpo_p1:>8.3f} {grpo_p8:>8.3f}")
print(f"{'delta':10} {grpo_p1-base_p1:>+8.3f} {grpo_p8-base_p8:>+8.3f}")

json.dump({"base_p1":base_p1,"base_p8":base_p8,
           "grpo7k_p1":grpo_p1,"grpo7k_p8":grpo_p8,
           "scope":"full dev 1034, loose exec_match, K=8, temp=0.8"},
          open(f"{SAVE_DIR}/passk_base_vs_grpo7k.json","w"), indent=2)

             pass@1   pass@8
base          0.496    0.733
GRPO 7K       0.677    0.809
delta        +0.181   +0.075


In [46]:
import shutil, json, datetime, os
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = f"/content/drive/MyDrive/spider_grpo_fullrun/{stamp}"
os.makedirs(SAVE_DIR, exist_ok=True)

# prediction files (re-scorable anytime, no GPU)
for f in ["/content/gold.txt", "/content/pred.txt"]:
    if os.path.exists(f): shutil.copy(f, SAVE_DIR)

# raw generations — the expensive part
json.dump(grpo_greedy, open(f"{SAVE_DIR}/grpo7k_greedy.json","w"))
if "base_greedy" in globals(): json.dump(base_greedy, open(f"{SAVE_DIR}/base_greedy.json","w"))
if "base_preds"  in globals():
    with open(f"{SAVE_DIR}/pred_base.txt","w") as f: f.write("\n".join(base_preds)+"\n")
if "base_samp"   in globals(): json.dump(base_samp, open(f"{SAVE_DIR}/base_samples_k8.json","w"))

summary = {
  "model": MODEL_ID, "n_train": 7000, "n_dev": len(examples),
  "EX_official": {"easy":0.847,"medium":0.455,"hard":0.408,"extra":0.223,"all":0.504},
  "exact_match_all": 0.423,
  "IUEN_acc": {"hard":0.143,"extra":0.059,"all":0.105},
  "ts_eval_single_db": 0.697, "ts_eval_testsuite_db": 0.607, "fp_gap": 0.090,
  "passk_loose_fulldev": {
      "grpo7k_p1": grpo_p1, "grpo7k_p8": grpo_p8,
      "base_p1": globals().get("base_p1"), "base_p8": globals().get("base_p8")},
  "prior_runs": {"base_EX": 0.345, "grpo500_EX": 0.434,
                 "grpo500_ts_single": 0.601, "grpo500_ts_suite": 0.519},
  "timestamp": stamp,
}
json.dump(summary, open(f"{SAVE_DIR}/summary.json","w"), indent=2)
print(SAVE_DIR); print(json.dumps(summary, indent=2))

/content/drive/MyDrive/spider_grpo_fullrun/20260723_073136
{
  "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "n_train": 7000,
  "n_dev": 1034,
  "EX_official": {
    "easy": 0.847,
    "medium": 0.455,
    "hard": 0.408,
    "extra": 0.223,
    "all": 0.504
  },
  "exact_match_all": 0.423,
  "IUEN_acc": {
    "hard": 0.143,
    "extra": 0.059,
    "all": 0.105
  },
  "ts_eval_single_db": 0.697,
  "ts_eval_testsuite_db": 0.607,
  "fp_gap": 0.09,
  "passk_loose_fulldev": {
    "grpo7k_p1": 0.6769825918762089,
    "grpo7k_p8": 0.8085106382978723,
    "base_p1": 0.4961315280464217,
    "base_p8": 0.7330754352030948
  },
  "prior_runs": {
    "base_EX": 0.345,
    "grpo500_EX": 0.434,
    "grpo500_ts_single": 0.601,
    "grpo500_ts_suite": 0.519
  },
  "timestamp": "20260723_073136"
}
